In [2]:
# bge-m3模型
from milvus_model.hybrid import BGEM3EmbeddingFunction

D:\envs\anaconda3\envs\edu_qa_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
m3_path = "../rag_qa/models/bge-m3"
embedding_function = BGEM3EmbeddingFunction(model_name_or_path=m3_path, use_fp16=False, device='cpu')

In [4]:
help(embedding_function)

Help on BGEM3EmbeddingFunction in module milvus_model.hybrid.bge_m3 object:

class BGEM3EmbeddingFunction(milvus_model.base.BaseEmbeddingFunction)
 |  BGEM3EmbeddingFunction(model_name: str = 'BAAI/bge-m3', batch_size: int = 16, device: str = None, normalize_embeddings: bool = True, use_fp16: bool = False, return_dense: bool = True, return_sparse: bool = True, return_colbert_vecs: bool = False, **kwargs)
 |  
 |  Method resolution order:
 |      BGEM3EmbeddingFunction
 |      milvus_model.base.BaseEmbeddingFunction
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __call__(self, texts: List[str]) -> Dict
 |  
 |  __init__(self, model_name: str = 'BAAI/bge-m3', batch_size: int = 16, device: str = None, normalize_embeddings: bool = True, use_fp16: bool = False, return_dense: bool = True, return_sparse: bool = True, return_colbert_vecs: bool = False, **kwargs)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  encode_documents(self, documents: Li

In [5]:
embed = embedding_function(["黑马JAVA", "黑马大模型", "黑马大模型"])
embed

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


{'dense': [array([-0.03068057,  0.00156492, -0.04899889, ...,  0.04231707,
          0.01612832,  0.07133269], dtype=float32),
  array([-0.01287241, -0.01234256, -0.06556934, ...,  0.04092155,
         -0.01335395,  0.0563211 ], dtype=float32),
  array([-0.01287241, -0.01234256, -0.06556934, ...,  0.04092155,
         -0.01335395,  0.0563211 ], dtype=float32)],
 'sparse': <3x250002 sparse array of type '<class 'numpy.float64'>'
 	with 14 stored elements in Compressed Sparse Row format>}

In [6]:
embed["dense"]

[array([-0.03068057,  0.00156492, -0.04899889, ...,  0.04231707,
         0.01612832,  0.07133269], dtype=float32),
 array([-0.01287241, -0.01234256, -0.06556934, ...,  0.04092155,
        -0.01335395,  0.0563211 ], dtype=float32),
 array([-0.01287241, -0.01234256, -0.06556934, ...,  0.04092155,
        -0.01335395,  0.0563211 ], dtype=float32)]

In [7]:
# row = embed["sparse"][0]

try:
    # 新版本 milvus-model 使用 coo_array 格式
    row = embed["sparse"][0]
    if hasattr(row, 'col'):  # coo_array 格式
        indices = row.col
        values = row.data
    else:  # csr_matrix 格式
        indices = row.indices
        values = row.data
except Exception as e:
    # 兼容旧版本 milvus-model
    row = embed["sparse"].getrow(0)
    indices = row.indices
    values = row.data

print(indices)
print(values)

[     6   5958   7320 189950]
[0.00417873 0.24346846 0.1504699  0.28743055]


C:\Users\matef\AppData\Local\Temp\ipykernel_19116\3903545189.py:14: DeprecationWarning: `getrow` is deprecated and will be removed in v1.14.0; use `X[[0]]` instead.
  row = embed["sparse"].getrow(0)


In [11]:
#row.col
row.indices

array([     6,   5958,   7320, 189950])

In [12]:
row.data

array([0.00417873, 0.24346846, 0.1504699 , 0.28743055])

In [14]:
# reranker 模型
from sentence_transformers import CrossEncoder

In [15]:
reranker_path = "../rag_qa/models/bge-reranker-large"
reranker = CrossEncoder(reranker_path)

In [16]:
queries = ["天气不错哈"]
documents = [
    "深度学习是机器学习的一个分支。",
    "今天的天气很好。",
    "机器学习是一种让计算机从数据中自动学习规律的方法。",
]
# 输入内容是文本对
pairs = [[queries[0], doc] for doc in documents]
print(pairs)


[['天气不错哈', '深度学习是机器学习的一个分支。'], ['天气不错哈', '今天的天气很好。'], ['天气不错哈', '机器学习是一种让计算机从数据中自动学习规律的方法。']]


In [17]:
scores = reranker.predict(pairs)
print(f'scores-->{scores}')


# 输出排序结果
for doc, score in sorted(zip(documents, scores), key=lambda x: x[1], reverse=True):
    print(f"{score:.4f} - {doc}")

scores-->[7.628980e-05 9.910616e-01 7.630377e-05]
0.9911 - 今天的天气很好。
0.0001 - 机器学习是一种让计算机从数据中自动学习规律的方法。
0.0001 - 深度学习是机器学习的一个分支。


In [18]:
reranker.predict(["我喜欢这个苹果", "我觉得apple很好！"])
# reranker.predict(["我喜欢这个苹果", "我不喜欢这个苹果！"])

0.9838244